In [3]:
import os
import sys
sys.path.append('../src')

import pandas as pd

from preprocessing import *
from models import *

In [4]:
project_path = os.path.dirname(os.getcwd())
dfs = get_dfs(project_path)
print(dfs['baseline_parameters'].columns)
nan_counts = dfs['baseline_parameters'].isna().sum()
nan_counts

Index(['PatientID', 'TransplantationID', 'SpenderID', 'Geburtsdatum',
       'Todesdatum', 'Datum', 'Geschlecht', 'ET_Grunderkrankung',
       'Grunderkrankung', 'Datum_erste_Dialyse', 'Dialyseart', 'Anzahl',
       'Alter', 'Ort', 'Blutgruppe', 'Koerpergroesse', 'CMV_AK', 'HbS_AG',
       'HCV_AK', 'anti_HIV', 'EBV_IgG', 'delayed graft function_inverse',
       'Dialyse_Anzahl', 'cold ischemia time', 'MMA_broad', 'MMB_broad',
       'MMDR_broad', 'MM_broad', 'Date of graft loss', 'Loss cause',
       'Programm', 'MaxvonEnddatum', 'SOURCE', 'Organqualitaet', 'rec_height',
       '1', '2', '3', '4', 's_Blutgruppe', 'R Full Phenotype', 'D-pheno',
       'PIRCHE_Score', 'RETNR'],
      dtype='object')


PatientID                            0
TransplantationID                    0
SpenderID                           47
Geburtsdatum                         0
Todesdatum                        2477
Datum                                0
Geschlecht                           0
ET_Grunderkrankung                 632
Grunderkrankung                    462
Datum_erste_Dialyse                279
Dialyseart                         583
Anzahl                             561
Alter                                0
Ort                                  0
Blutgruppe                          52
Koerpergroesse                      82
CMV_AK                             165
HbS_AG                             118
HCV_AK                             118
anti_HIV                          2464
EBV_IgG                           2321
delayed graft function_inverse     147
Dialyse_Anzahl                     785
cold ischemia time                 138
MMA_broad                          128
MMB_broad                

In [7]:
# HLA information
for col in ['MMA_broad', 'MMB_broad', 'MMDR_broad', 'MM_broad']:
    print(f"{col} - Value Counts:\n{dfs['baseline_parameters'][col].value_counts()}\nNaN Count: {dfs['baseline_parameters'][col].isna().sum()}\n")


MMA_broad - Value Counts:
MMA_broad
1.0     1651
0.0     1209
2.0      583
14.0       1
Name: count, dtype: int64
NaN Count: 128

MMB_broad - Value Counts:
MMB_broad
1.0    1588
2.0    1050
0.0     805
3.0       1
Name: count, dtype: int64
NaN Count: 128

MMDR_broad - Value Counts:
MMDR_broad
1.0    1621
0.0    1128
2.0     694
3.0       1
Name: count, dtype: int64
NaN Count: 128

MM_broad - Value Counts:
MM_broad
3.0    878
2.0    708
4.0    560
0.0    489
5.0    423
1.0    240
6.0    146
Name: count, dtype: int64
NaN Count: 128



In [ ]:
filtered_df = dfs['biopsy'].dropna(subset=['Banff 17 categorie'])

# Count occurrences per patient
banff_counts_per_patient = filtered_df.groupby('PatientID')['Banff 17 categorie'].value_counts().unstack(fill_value=0)
print(banff_counts_per_patient)
dfs['biopsy']['Banff 17 categorie'].value_counts()
print(f"unique patients {filtered_df['PatientID'].nunique()}")

Banff 17 categorie  1.0  2.0  3.0  4.0  5.0  6.0
PatientID                                       
35                    0    0    0    0    1    0
418                   0    0    0    0    0    3
712                   0    0    0    0    0    1
1054                  0    0    1    0    0    3
1148                  0    0    0    0    0    1
...                 ...  ...  ...  ...  ...  ...
13283                 0    0    0    2    0    1
13558                 0    0    0    0    1    0
14363                 0    0    0    1    0    0
25131                 0    0    1    1    0    4
34613                 0    0    0    0    2    0

[836 rows x 6 columns]
unique patients 836


Banff 17 categorie
6.0    949
4.0    321
3.0    266
5.0    194
1.0    104
2.0     67
Name: count, dtype: int64

In [3]:
print(dfs['baseline_parameters']['PatientID'].nunique())
print(dfs['baseline_parameters']['TransplantationID'].nunique())

3474
3570


In [4]:
print(dfs['donorparameters'].columns)
nan_counts = dfs['donorparameters'].isna().sum()
nan_counts

Index(['PatientID', 'TransplantationID', 'SpenderID', 'gender_donor',
       'age_donor', 'donor_bloodgroup', 'type_of_donation', 'donor_COD',
       'donor_weight', 'donor_height', 'SOURCE'],
      dtype='object')


PatientID              0
TransplantationID      0
SpenderID             49
gender_donor          49
age_donor             74
donor_bloodgroup     110
type_of_donation      62
donor_COD            198
donor_weight         151
donor_height         157
SOURCE                 0
dtype: int64

In [5]:
# processing static features of patients

print(f"transplantations in baseline: {dfs['baseline_parameters']['PatientID'].count()}, unique patients: {dfs['baseline_parameters']['PatientID'].nunique()}\n")

print("remove patients where patientID, spenderID or transplantationID is NaN")
static_df = dfs['baseline_parameters'][['PatientID', 'TransplantationID', 'SpenderID', 'Geburtsdatum', 'Todesdatum', 'Datum', 'Geschlecht', 'Grunderkrankung', 'Datum_erste_Dialyse', 'Dialyse_Anzahl', 'Alter', 'Blutgruppe', 'Koerpergroesse', 'Date of graft loss', 'Loss cause', 'cold ischemia time']].rename(
    columns={
        'PatientID': 'patient_id',
        'SpenderID': 'donor_id',
        'TransplantationID': 'transplant_id',
        'Geburtsdatum': 'birth_date',
        'Todesdatum': 'death_date',
        'Datum': 'transplant_date',
        'Geschlecht': 'gender',
        'Grunderkrankung': 'underlying_disease',
        'Datum_erste_Dialyse': 'first_dialysis_date',
        'Dialyse_Anzahl': 'number_dialyses',
        'Alter': 'age',
        'Blutgruppe': 'blood_group',
        'Koerpergroesse': 'height',
        'Date of graft loss': 'loss_date',
        'Loss cause': 'loss_cause',
        'cold ischemia time': 'cold_ischemia_time',
    }
)
static_df = static_df.dropna(subset=['patient_id', 'donor_id', 'transplant_id'])

# remove duplicate patient ids
static_df = static_df.drop_duplicates(subset='patient_id', keep='first')

print(f"transplantations in baseline: {static_df['patient_id'].count()}, unique patients: {static_df['patient_id'].nunique()}")
print(static_df.columns)

transplantations in baseline: 3572, unique patients: 3474

remove patients where patientID, spenderID or transplantationID is NaN
transplantations in baseline: 3433, unique patients: 3433
Index(['patient_id', 'transplant_id', 'donor_id', 'birth_date', 'death_date',
       'transplant_date', 'gender', 'underlying_disease',
       'first_dialysis_date', 'number_dialyses', 'age', 'blood_group',
       'height', 'loss_date', 'loss_cause', 'cold_ischemia_time'],
      dtype='object')


In [6]:
print("match transplantation patients to their donors")
static_df = static_df.merge(
    dfs['donorparameters'][['PatientID', 'TransplantationID', 'SpenderID', 'gender_donor', 'age_donor', 'donor_bloodgroup',
                            'type_of_donation', 'donor_COD', 'donor_weight', 'donor_height', 'SOURCE']],
    how='left',  # Left join to keep all records in static_df
    left_on=['patient_id', 'transplant_id', 'donor_id'],
    right_on=['PatientID', 'TransplantationID', 'SpenderID']
)

static_df = static_df.drop(columns=['PatientID', 'TransplantationID', 'SpenderID'])

print(f"transplantations in baseline: {static_df['patient_id'].count()}, unique patients: {static_df['patient_id'].nunique()}")
print(static_df.columns)
print(static_df.isna().sum())

match transplantation patients to their donors
transplantations in baseline: 3433, unique patients: 3433
Index(['patient_id', 'transplant_id', 'donor_id', 'birth_date', 'death_date',
       'transplant_date', 'gender', 'underlying_disease',
       'first_dialysis_date', 'number_dialyses', 'age', 'blood_group',
       'height', 'loss_date', 'loss_cause', 'cold_ischemia_time',
       'gender_donor', 'age_donor', 'donor_bloodgroup', 'type_of_donation',
       'donor_COD', 'donor_weight', 'donor_height', 'SOURCE'],
      dtype='object')
patient_id                0
transplant_id             0
donor_id                  0
birth_date                0
death_date             2372
transplant_date           0
gender                    0
underlying_disease      416
first_dialysis_date     260
number_dialyses         712
age                       0
blood_group              33
height                   52
loss_date              2811
loss_cause             2846
cold_ischemia_time       92
gender_donor 

In [7]:
static_df['transplant_date'] = pd.to_datetime(static_df['transplant_date'], dayfirst=True, errors='coerce')
static_df['loss_date'] = pd.to_datetime(static_df['loss_date'], dayfirst=True, errors='coerce')
static_df['death_date'] = pd.to_datetime(static_df['death_date'], dayfirst=True, errors='coerce')

# if loss date is given calculate the relative days from transplantation date
static_df['loss_rel_days'] = (static_df['loss_date'] - static_df['transplant_date']).dt.days
static_df['death_rel_days'] = (static_df['death_date'] - static_df['transplant_date']).dt.days

In [8]:
# selecting relevant features
static_df = static_df[
    [
        'patient_id',
        'transplant_id',
        'transplant_date',
        'gender',
        'age',
        'death_rel_days',
        'underlying_disease',
        'number_dialyses',
        'blood_group',
        'height',
        'loss_rel_days',
        'loss_cause',
        'cold_ischemia_time',
        'gender_donor',
        'age_donor',
        'donor_bloodgroup',
        'type_of_donation',
        'donor_height',
        'donor_weight'
    ]
]

In [2]:
# creating static df using the function
project_path = os.path.dirname(os.getcwd())
dfs = get_dfs(project_path)
stat = create_static_df(dfs)
stat

,patient_id,transplant_id,transplant_date,gender,age,underlying_disease,number_dialyses,blood_group,height,death_rel_days,...,age_donor,donor_bloodgroup,type_of_donation,donor_height,donor_weight,pirche_score,mma_broad,mmb_broad,mmdr_broad,mm_broad
0,33,1805,2003-12-19,m,39.54,"Pyelonephritis, chron.",0.0,A+,171.0,2101.0,...,33.0,A+,hirntot,175.0,70.0,67.1308,1,1,2,4
1,35,2661,2009-07-07,m,66.33,Zystennieren,2.0,B+,180.0,2784.0,...,66.0,0,hirntot,184.0,83.0,36.4600,2,2,1,5
2,67,3199,2019-02-13,m,53.01,Proliferative GN,1.0,0+,180.0,NaN,...,64.0,0+,hirntot,175.0,100.0,87.1523,0,1,2,3
3,83,4490,2013-11-13,m,55.39,mesangio proliferative Glomerulonephritis,0.0,0+,174.0,NaN,...,53.0,0-,lebend,174.0,62.0,125.5263,1,2,1,4
4,87,1702,2006-06-16,w,49.09,IgA Nephropathie,0.0,0+,163.0,4275.0,...,49.0,0+,hirntot,184.0,98.0,48.0467,1,2,1,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3520,38051,15267,2019-05-06,w,68.64,NaN,NaN,NaN,NaN,NaN,...,78.0,NaN,NaN,180.0,80.0,59.6151,<NA>,<NA>,<NA>,<NA>
3521,38056,15359,2015-06-29,m,40.85,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,lebend,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>
3522,38071,15166,2017-02-01,w,48.32,NaN,NaN,NaN,NaN,NaN,...,56.0,NaN,lebend,NaN,80.0,185.7676,<NA>,<NA>,<NA>,<NA>
3523,38081,15319,2013-06-17,m,50.60,NaN,NaN,NaN,NaN,NaN,...,55.0,A+,lebend,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>
